# Passenger Generation Map Notebook

This notebook implements a pedestrian volume estimation model based on traffic data and generates passenger spawning points for transportation simulation.

## Overview
- **Section 1**: Setup and coefficients for the pedestrian volume model
- **Section 2**: Data input and loading
- **Section 3**: Execution of the model and demo with 10k points
- **Section 4**: Visualization of results

The model uses Equation 2 from the paper to estimate pedestrian volume (v_ped) based on traffic intensity, building density, and betweenness centrality.

In [1]:
# =================================================================
# 1. SETUP: MODEL COEFFICIENTS (Section 3.1.1)
# =================================================================
# Note: These beta values should be calibrated based on local 
# sample counts or transferred from similar studies.

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from folium.plugins import HeatMap

BETAS = {
    'beta_0': 0.5,   # Intercept
    'beta_1': 0.6,   # ln(ADT_prop) - Traffic Intensity 
    'beta_2': 0.3,   # ln(D_bldg) - Building Density 
    'beta_3': 0.2,   # ln(C_B) - Betweenness Centrality 
    'beta_4': 0.1,   # L_mix - Land-use mix (optional) 
    'epsilon': 0.05  # Error term 
}

def calculate_v_ped(df, betas):
    """
    Calculates the estimated daily pedestrian volume (V_ped) 
    using the log-linear regression model.
    """
    # We use a small epsilon (1e-6) to avoid errors with log(0)
    ln_v_ped = (
        betas['beta_0'] +
        betas['beta_1'] * np.log(df['ADT_prop'] + 1e-6) +
        betas['beta_2'] * np.log(df['bldg_density'] + 1e-6) +
        betas['beta_3'] * np.log(df['bc'] + 1e-6) +
        betas['epsilon']
    )
    # Return exponentiated result to get actual volume 
    return np.exp(ln_v_ped)

def generate_passenger_points(df, n_points=10000):
    """
    Generates passenger origins using a weighted spatial distribution.
    Edges with higher V_ped generate proportionally more demand.
    """
    # Normalize V_ped values to create a spatial probability distribution 
    weights = df['v_ped'] / df['v_ped'].sum()
    
    # Sample origins based on the distribution 
    sampled_indices = np.random.choice(df.index, size=n_points, p=weights)
    
    return df.loc[sampled_indices].copy()

In [2]:
# ==========================================
# 2. DATA INPUT (Where your file goes)
# ==========================================

from pathlib import Path

possible_paths = [
    Path('data/processed/nodes_with_tomtom_data_imputed.csv'),
    Path('..', 'data', 'processed', 'nodes_with_tomtom_data_imputed.csv'),
    Path('..', '..', 'data', 'processed', 'nodes_with_tomtom_data_imputed.csv')
]

csv_path = next((p for p in possible_paths if p.exists()), None)

if csv_path is None:
    print('File not found in expected locations:')
    for p in possible_paths:
        print('  -', p.resolve())
    print('Falling back to synthetic demo data...')
    data = {
        'base_osmid': range(1000, 1100),
        'lat': np.random.uniform(8.20, 8.25, 100),
        'lon': np.random.uniform(124.23, 124.28, 100),
        'bc': np.random.gamma(2, 2, 100),            # Structural importance
        'bldg_density': np.random.uniform(5, 100, 100), # Trip generation proxy
        'ADT_prop': np.random.uniform(100, 5000, 100)   # Observed traffic
    }
    df_city = pd.DataFrame(data)
else:
    print(f"Loaded actual traffic data from: {csv_path.resolve()}")
    df_city = pd.read_csv(csv_path, dtype={
        'base_osmid': 'int64',
        'lat': 'float64',
        'lon': 'float64',
        'bc': 'float64',
        'bldg_density': 'float64',
        'traffic_index': 'float64',
        'ADT_prop': 'float64'
    })


Loaded actual traffic data from: C:\Users\jradl\OneDrive\Documents\Thesis\Thesis\data\processed\nodes_with_tomtom_data_imputed.csv


In [3]:
# ==========================================
# 3. EXECUTION & DEMO (10k Points)
# ==========================================

# Step A: Calculate Pedestrian Volume (V_ped) for all edges
df_city['v_ped'] = calculate_v_ped(df_city, BETAS)

# Step B: Generate 10k "Passenger Spawning" events [cite: 35, 48]
passenger_demand = generate_passenger_points(df_city, n_points=100000)

In [4]:
# ==========================================
# 4. VISUALIZATION: Overlay Heatmap on Actual Map
# ==========================================

# Create a folium map centered on Iligan
center_lat = df_city['lat'].mean()
center_lon = df_city['lon'].mean()
m = folium.Map(location=[center_lat, center_lon], zoom_start=13, tiles='OpenStreetMap')

# Prepare data for heatmap: list of [lat, lon] for passenger generation points
heat_data = passenger_demand[['lat', 'lon']].values.tolist()

# Add heatmap layer for passenger generation likelihood
HeatMap(heat_data, radius=15, blur=10, max_zoom=1).add_to(m)

# Save the map to an HTML file
m.save('../results/passenger_generation_heatmap.html')
print("Map saved to: ../results/passenger_generation_heatmap.html")

# Display the map inline in the notebook
from IPython.display import HTML
HTML(m._repr_html_())

Map saved to: ../results/passenger_generation_heatmap.html
